# BIOTEQ Muscle Engine — Colab demo
Run **Runtime → Run all** on a CPU runtime. This notebook runs the Phase 1 `MuscleEngine` (`rules_v0.1`) on simulated bicep-curl data, displays derived metrics, plots sensor signals, and exports JSON.

This is a deterministic rule-based engine; no trained ML classifier exists yet. The demo threshold is arbitrary and does not diagnose injury. All example measurements and labels are simulated.

**Before using Colab:** commit and push the implementation and this notebook to GitHub, then open this `.ipynb` via Colab's GitHub tab. Changes that exist only on your computer are not available to Colab. For a private repository, upload a ZIP of your checkout using the option below; do not put access tokens in this notebook.

## 1. Locate the code and install dependencies
Choose `github` to clone the published repo or `upload_zip` to upload your local checkout (ZIP must include `bioteq/`, `requirements.txt`, and `examples/`; exclude `.venv` and secrets). Local Jupyter execution automatically reuses the checkout containing this notebook. No GPU is needed.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/EEsc05/BIOTEQ.git"
REPO_REF = ""  # Optional published branch or tag; blank uses the repository default.
SOURCE = "github"  # Change to "upload_zip" for unpublished/local changes.

candidates = [Path.cwd(), Path.cwd().parent, Path("/content/BIOTEQ")]
repo = next((p for p in candidates if (p / "bioteq/engine/muscle_engine.py").is_file()), None)
if repo is None:
    if SOURCE == "upload_zip":
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one repository ZIP.")
        destination = Path("/content/bioteq_upload").resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                if not (destination / item.filename).resolve().is_relative_to(destination):
                    raise ValueError("ZIP contains an unsafe path.")
            archive.extractall(destination)
        matches = list(destination.rglob("bioteq/engine/muscle_engine.py"))
        if len(matches) != 1:
            raise ValueError("ZIP must contain exactly one BIOTEQ implementation.")
        repo = matches[0].parents[2]
    elif SOURCE == "github":
        repo = Path("/content/BIOTEQ")
        if repo.exists():
            raise RuntimeError("/content/BIOTEQ exists but lacks the engine. Rename it or restart the runtime.")
        command = ["git", "clone", "--depth", "1"]
        if REPO_REF:
            command += ["--branch", REPO_REF]
        subprocess.check_call(command + [REPO_URL, str(repo)])
    else:
        raise ValueError("SOURCE must be github or upload_zip.")
repo = repo.resolve()
if not (repo / "bioteq/engine/muscle_engine.py").is_file():
    raise RuntimeError("The selected GitHub version lacks the engine. Push your changes or use upload_zip.")
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
# Set this variable only for local validation in an already prepared environment.
if os.environ.get("BIOTEQ_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib>=3.8,<4"])
print("Using repository:", repo)

## 2. Load a session and run the engine
The supplied fixture contains ten repetitions over 30 seconds, EMG sampled every 250 ms, motion every 100 ms, and heart rate/NIRS every second. Default rules produce no demo warning. A second pass explicitly enables the software demonstration threshold.

To use your own simulator CSV, change `CSV_PATH`. It must follow `SensorReading` and contain one session and exercise. Do not supply event labels as sensor inputs.

In [ ]:
import json
from IPython.display import display, HTML, FileLink
from bioteq.ingestion.reader import SensorReader
from bioteq.engine.muscle_engine import MuscleEngine

CSV_PATH = repo / "examples/sample_session.csv"
readings = SensorReader().read_csv(CSV_PATH)
output = MuscleEngine().process_session(readings)
demo_output = MuscleEngine(config_path=repo / "bioteq/config/demo_thresholds.yaml").process_session(readings)
print(f"Session: {output.session_id} | exercise: {output.exercise}")
print(f"Data source: {output.data_source} | engine: {output.model_version}")
print(f"Readings: {len(readings):,} | duration: {(output.end_ms-output.start_ms)/1000:g} s")

## 3. Derived metrics and rule outputs
Missing measurements display as unavailable. EMG normalization and NIRS baseline changes require explicit references, so they remain null in this default demo. Heart-rate recovery requires an explicit rest interval. The 38% planted difference uses the left side as denominator; symmetric imbalance uses the mean of both sides.

In [ ]:
from html import escape

def table(headers, rows):
    def cell(value):
        if value is None:
            return "unavailable"
        if isinstance(value, float):
            return f"{value:.3f}"
        return escape(str(value))
    display(HTML("<table><thead><tr>" + "".join(f"<th>{escape(h)}</th>" for h in headers) +
                 "</tr></thead><tbody>" + "".join("<tr>" + "".join(f"<td>{cell(v)}</td>" for v in row) +
                 "</tr>" for row in rows) + "</tbody></table>"))

m = output.metrics
table(["Metric", "Value", "Unit"], [
    ("Repetitions", m.movement.rep_count, "count"),
    ("Average peak-to-peak repetition duration", m.movement.average_rep_duration_s, "s"),
    ("Repetition duration variability", m.movement.movement_variability_cv, "CV"),
    ("Mean heart rate", m.cardiovascular.heart_rate_bpm, "bpm"),
    ("Heart-rate recovery", m.cardiovascular.heart_rate_recovery_bpm, "bpm"),
])
table(["Muscle", "Side", "RMS (mV)", "Mean absolute (mV)", "Normalized activation (%)"],
      [(x.muscle_site, x.side, x.rms_mv, x.mean_absolute_mv, x.normalized_activation_pct) for x in m.muscles])
table(["Muscle", "Symmetric imbalance (%)", "Right greater than left (%)"],
      [(x.muscle_site, x.imbalance_pct, x.right_greater_than_left_pct) for x in m.bilateral])
table(["Muscle", "Side", "Mean oxygenation (%)", "Baseline change (percentage points)"],
      [(x.muscle_site, x.side, x.smo2_pct, x.change_percentage_points) for x in m.oxygenation])
print("Quality flags:", output.quality_flags or "none")
print("Default rule warnings:", len(output.warnings))
print("Demo rule warnings:", len(demo_output.warnings))
for warning in demo_output.warnings:
    print(f"{warning.code}: {warning.message} Evaluated window: {warning.start_ms}–{warning.end_ms} ms.")

## 4. Plot the observed signals and derived EMG RMS
Motion magnitude uses complete x/y/z readings at identical timestamps. Sensor channels retain their original timestamps; no missing values are interpolated. The fixture's constant EMG, heart-rate, and NIRS signals are deliberate software inputs, not realistic physiological waveforms. Rep counting depends on gravity, orientation, and exercise.

In [ ]:
from collections import defaultdict
from math import sqrt
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
fig.suptitle(f"BIOTEQ — {output.session_id} ({output.data_source})")

def signal_plot(ax, channel, unit, ylabel):
    groups = defaultdict(list)
    for r in sorted(readings, key=lambda r: r.timestamp_ms):
        if r.channel == channel and r.unit == unit and r.value is not None:
            groups[(r.muscle_site, r.side)].append((r.timestamp_ms/1000, r.value))
    for (site, side), points in groups.items():
        times, values = zip(*points)
        ax.plot(times, values, label=" ".join(x for x in (site, side) if x) or channel.upper())
    if groups:
        ax.legend(fontsize=8)
    else:
        ax.text(.5, .5, "No compatible measurements", ha="center", transform=ax.transAxes)
    ax.set(xlabel="Time (s)", ylabel=ylabel)
    ax.grid(alpha=.25)

signal_plot(axes[0, 0], "emg", "mV", "EMG amplitude (mV)")
axes[0, 0].set_title("Observed EMG")
frames = defaultdict(dict)
for r in readings:
    if r.channel == "accelerometer" and r.value is not None:
        frames[r.timestamp_ms][r.axis] = r.value
motion = sorted((t/1000, sqrt(sum(v[a]**2 for a in ("x", "y", "z"))))
                for t, v in frames.items() if all(a in v for a in ("x", "y", "z")))
if motion:
    times, values = zip(*motion)
    axes[0, 1].plot(times, values, color="tab:orange")
else:
    axes[0, 1].text(.5, .5, "No complete XYZ frames", ha="center", transform=axes[0, 1].transAxes)
axes[0, 1].set(title=f"Acceleration magnitude — {m.movement.rep_count} detected reps",
               xlabel="Time (s)", ylabel="Magnitude (m/s²)")
axes[0, 1].grid(alpha=.25)
signal_plot(axes[1, 0], "ecg", "bpm", "Heart rate (bpm)")
axes[1, 0].set_title("Supplied heart rate")
signal_plot(axes[1, 1], "nirs", "percent", "Muscle oxygen saturation (%)")
axes[1, 1].set_title("Observed NIRS")
plt.show()

if m.muscles:
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.bar([f"{x.muscle_site}\n{x.side or 'unpaired'}" for x in m.muscles],
           [x.rms_mv for x in m.muscles], color=["tab:blue" if x.side == "left" else "tab:orange" for x in m.muscles])
    ax.set(title="Derived whole-window EMG RMS", ylabel="RMS amplitude (mV)")
    ax.grid(axis="y", alpha=.25)
    plt.show()

## 5. Inspect and export structured output
The default and demo results remain separate. Warning bounds describe the evaluated window, not a localized onset. `FileLink` works in Jupyter; the optional download cell uses Colab's download interface.

In [ ]:
print(output.model_dump_json(indent=2))
export_dir = repo / "notebook_outputs"
export_dir.mkdir(exist_ok=True)
for name, result in [("session_output.json", output), ("demo_output.json", demo_output)]:
    path = export_dir / name
    path.write_text(result.model_dump_json(indent=2) + "\n")
    display(FileLink(str(path)))

In [ ]:
DOWNLOAD_JSON = False  # Set True to download both result files in Colab.
if DOWNLOAD_JSON:
    from google.colab import files
    for name in ("session_output.json", "demo_output.json"):
        files.download(str(export_dir / name))

## 6. Sample-fixture checks
These checks verify the deterministic software example only. They are not model accuracy scores or public-data validation. Checks skip automatically for a different CSV.

In [ ]:
if CSV_PATH.resolve() == (repo / "examples/sample_session.csv").resolve():
    import math
    assert output.metrics.movement.rep_count == 10
    assert math.isclose(output.metrics.movement.average_rep_duration_s, 3.0)
    trapezius = next(x for x in m.bilateral if x.muscle_site == "trapezius")
    assert math.isclose(trapezius.right_greater_than_left_pct, 38.0)
    assert not output.warnings
    assert any(w.code == "possible_compensation" for w in demo_output.warnings)
    print("Passed: 10 repetitions, 3-second cadence, 38% right-side difference, and demo-only rule flag.")
else:
    print("Custom session: sample-specific assertions skipped.")